In [1]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.models import ResNetFineTuned
from src.train import train
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights, IMAGENET_MEAN, IMAGENET_STD, DX_TO_IDX, IDX_TO_DX
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

Read in training and validation data, initialize HAM10000Dataset objects for each

In [2]:
train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')

image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')

train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

Build ResNet-18 pretrained model and look at parameters to visualize options for unfreezing

In [3]:
resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for name, param in resnet_model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")

Layer: conv1.weight | Size: torch.Size([64, 3, 7, 7])
Layer: bn1.weight | Size: torch.Size([64])
Layer: bn1.bias | Size: torch.Size([64])
Layer: layer1.0.conv1.weight | Size: torch.Size([64, 64, 3, 3])
Layer: layer1.0.bn1.weight | Size: torch.Size([64])
Layer: layer1.0.bn1.bias | Size: torch.Size([64])
Layer: layer1.0.conv2.weight | Size: torch.Size([64, 64, 3, 3])
Layer: layer1.0.bn2.weight | Size: torch.Size([64])
Layer: layer1.0.bn2.bias | Size: torch.Size([64])
Layer: layer1.1.conv1.weight | Size: torch.Size([64, 64, 3, 3])
Layer: layer1.1.bn1.weight | Size: torch.Size([64])
Layer: layer1.1.bn1.bias | Size: torch.Size([64])
Layer: layer1.1.conv2.weight | Size: torch.Size([64, 64, 3, 3])
Layer: layer1.1.bn2.weight | Size: torch.Size([64])
Layer: layer1.1.bn2.bias | Size: torch.Size([64])
Layer: layer2.0.conv1.weight | Size: torch.Size([128, 64, 3, 3])
Layer: layer2.0.bn1.weight | Size: torch.Size([128])
Layer: layer2.0.bn1.bias | Size: torch.Size([128])
Layer: layer2.0.conv2.weight 

Construct Loss Function

In [4]:
class_weights = compute_class_weights(train_df)

loss_func = torch.nn.CrossEntropyLoss(weight=class_weights)

Create model from ResNet-18 pre-trained model with a backbone, with layer 4 of ResNet-18 and final classifier linear layer unfrozen 

In [5]:
model = ResNetFineTuned(["layer4", "fc"])

Frozen:  conv1.weight
Frozen:  bn1.weight
Frozen:  bn1.bias
Frozen:  layer1.0.conv1.weight
Frozen:  layer1.0.bn1.weight
Frozen:  layer1.0.bn1.bias
Frozen:  layer1.0.conv2.weight
Frozen:  layer1.0.bn2.weight
Frozen:  layer1.0.bn2.bias
Frozen:  layer1.1.conv1.weight
Frozen:  layer1.1.bn1.weight
Frozen:  layer1.1.bn1.bias
Frozen:  layer1.1.conv2.weight
Frozen:  layer1.1.bn2.weight
Frozen:  layer1.1.bn2.bias
Frozen:  layer2.0.conv1.weight
Frozen:  layer2.0.bn1.weight
Frozen:  layer2.0.bn1.bias
Frozen:  layer2.0.conv2.weight
Frozen:  layer2.0.bn2.weight
Frozen:  layer2.0.bn2.bias
Frozen:  layer2.0.downsample.0.weight
Frozen:  layer2.0.downsample.1.weight
Frozen:  layer2.0.downsample.1.bias
Frozen:  layer2.1.conv1.weight
Frozen:  layer2.1.bn1.weight
Frozen:  layer2.1.bn1.bias
Frozen:  layer2.1.conv2.weight
Frozen:  layer2.1.bn2.weight
Frozen:  layer2.1.bn2.bias
Frozen:  layer3.0.conv1.weight
Frozen:  layer3.0.bn1.weight
Frozen:  layer3.0.bn1.bias
Frozen:  layer3.0.conv2.weight
Frozen:  layer

Construct Optimizer

In [6]:
head_lr = 1e-3
backbone_lr = 1e-4
momentum = 0.9
adaptive_learning_rate = 0.999
betas = (momentum, adaptive_learning_rate)
complete_unfrozen_layers = model.get_complete_unfrozen_layers()
head_layer = model.get_head()
optimizer = torch.optim.AdamW(
    [
        {"params": complete_unfrozen_layers, "lr": backbone_lr},
        {"params": head_layer, "lr": head_lr}
    ],
    betas=betas
)

Train model

In [7]:
model = train(
    model=model,
    model_name="ResNetFineTuned-Layer4Unfrozen",
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=loss_func,
    optimizer=optimizer,
    num_epoch=100
)

Using CUDA
Epoch  1 / 100: train_acc=0.5977 val_acc=0.6996 train_f1=0.4154 val_f1=0.5331 
Epoch 10 / 100: train_acc=0.8056 val_acc=0.7250 train_f1=0.7384 val_f1=0.6205 
Epoch 20 / 100: train_acc=0.8462 val_acc=0.7747 train_f1=0.7762 val_f1=0.6343 
Epoch 30 / 100: train_acc=0.9010 val_acc=0.7906 train_f1=0.8847 val_f1=0.6520 
Epoch 40 / 100: train_acc=0.9219 val_acc=0.7879 train_f1=0.8928 val_f1=0.6592 
Epoch 50 / 100: train_acc=0.9405 val_acc=0.7901 train_f1=0.9131 val_f1=0.6667 
Epoch 60 / 100: train_acc=0.9470 val_acc=0.7882 train_f1=0.9291 val_f1=0.6066 
Epoch 70 / 100: train_acc=0.9539 val_acc=0.7880 train_f1=0.9327 val_f1=0.6493 
Epoch 80 / 100: train_acc=0.9633 val_acc=0.8051 train_f1=0.9560 val_f1=0.6641 
Epoch 90 / 100: train_acc=0.9589 val_acc=0.7993 train_f1=0.9519 val_f1=0.6793 
Epoch 100 / 100: train_acc=0.9629 val_acc=0.8000 train_f1=0.9449 val_f1=0.6657 
Model saved to logs\ResNetFineTuned-Layer4Unfrozen_0708_010323\ResNetFineTuned-Layer4Unfrozen.th
